# Algorithm Comparison: Logistic Regression vs Random Forest

This notebook compares two supervised learning algorithms for heart risk prediction.

**Selection rule:** prioritize **recall**, then use **F1-score** as tiebreaker.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_loader import load_heart_dataset

In [ ]:
X, y = load_heart_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

log_reg = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)
rf = Pipeline(
    steps=[
        ("model", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)),
    ]
)

models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
}

In [ ]:
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    metrics = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1_score": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probs) if probs is not None else None,
        "confusion_matrix": confusion_matrix(y_test, preds),
    }
    return metrics


rows = []
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    results[name] = {"model": model, **metrics}
    rows.append(
        {
            "Model": name,
            "Accuracy": metrics["accuracy"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-score": metrics["f1_score"],
            "ROC-AUC": metrics["roc_auc"],
        }
    )

comparison_df = pd.DataFrame(rows).sort_values(by=["Recall", "F1-score"], ascending=False)
comparison_df.style.format({"Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}", "F1-score": "{:.4f}", "ROC-AUC": "{:.4f}"})

In [ ]:
# 5-fold cross-validation comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, model in models.items():
    cv_scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    )
    cv_rows.append(
        {
            "Model": name,
            "CV Accuracy": cv_scores["test_accuracy"].mean(),
            "CV Precision": cv_scores["test_precision"].mean(),
            "CV Recall": cv_scores["test_recall"].mean(),
            "CV F1": cv_scores["test_f1"].mean(),
            "CV ROC-AUC": cv_scores["test_roc_auc"].mean(),
        }
    )

cv_df = pd.DataFrame(cv_rows).sort_values(by=["CV Recall", "CV F1"], ascending=False)
cv_df.style.format({
    "CV Accuracy": "{:.4f}",
    "CV Precision": "{:.4f}",
    "CV Recall": "{:.4f}",
    "CV F1": "{:.4f}",
    "CV ROC-AUC": "{:.4f}",
})

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, data) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=data["confusion_matrix"], display_labels=["Not At Risk", "At Risk"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# Final model decision based on recall-first logic
ordered = comparison_df.sort_values(by=["Recall", "F1-score"], ascending=False).reset_index(drop=True)
best_model_name = ordered.loc[0, "Model"]
best_recall = ordered.loc[0, "Recall"]
best_f1 = ordered.loc[0, "F1-score"]

print(f"Selected best model: {best_model_name}")
print(f"Recall: {best_recall:.4f} | F1-score: {best_f1:.4f}")
print("Reason: recall is prioritized to reduce missed at-risk cases.")